In [1]:
# task: review and allocate all data combinations within SMILES -> PDB
# 30k ish SMILES
# for each SMILES: many PDBs
# for each of those PDBs -> many context coordinates + Y
# need an allocator to generate training + test datasets for this task


# projected unpacking time = 41 minutes
# TODO: organize into train and test + batch processor

import torch
import re
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


class NAAnoPointer:

    def __init__(self, scaffoldpdb, pdb_radial, smiles_molfp,
                 train_pct, seed):
        scaffoldpdb["PDB_hits"] = scaffoldpdb["PDB_hits"].apply(ast.literal_eval)
        self.scaffoldpdb = scaffoldpdb
        self.radial_data = pdb_radial
        self.molfp_data = smiles_molfp

        if not 0 < train_pct < 1:
            raise ValueError(f"Train % split must be between 0.00 and 1.00: value submitted {train_pct}")
        self.train_pct = train_pct
        self.seed = seed

        self.train_rows = None
        self.test_rows = None


# notes on data columns
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'

    def make_pointerfile(self, outpath):
        rng = np.random.default_rng(self.seed)

        all_smiles = self.scaffoldpdb["SMILES"].tolist()
        rng.shuffle(all_smiles)

        n_train = int(len(all_smiles) * self.train_pct)
        train_set = set(all_smiles[:n_train])
        test_set = set(all_smiles[n_train:])  # unseen smiles -> ZeroBind methodology

        train_rows, test_rows = [], []

        for _, row in tqdm(self.scaffoldpdb.iterrows(), total=len(self.scaffoldpdb), desc="Building Pointer Files"):
            smiles = row["SMILES"]
            pdb_hits = row["PDB_hits"]
            bucket = train_rows if smiles in train_set else test_rows

            for pdb_id in pdb_hits:
                seq_data = self.radial_data.loc[self.radial_data["PDB_ID"]==pdb_id]
                if not len(seq_data):
                    continue
                seq_len = len(seq_data["radial_sequence"].values[0]) - 1   # remove the void token
                # this is because we want naanobot to predict the biochemical environment
                # skeleton is handling the "VOID" positions already
                n_windows = seq_len

                for window_idx in range(n_windows):
                    bucket.append({
                        "SMILES": smiles,
                        "PDB_HIT": pdb_id,
                        "WINDOW_IDX": window_idx,
                    })


        self.train_rows = pd.DataFrame(train_rows)
        self.test_rows = pd.DataFrame(test_rows)

        self.train_rows.to_parquet(outpath / "naanobot_training_pointers.parquet", index=False)
        self.test_rows.to_parquet(outpath / "naanobot_test_pointers.parquet", index=False)

        return True

# TODO: function to sample some molecular fingerprints and calculate tanimoto similarity (1000 at a time?)


In [2]:
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

In [3]:
scaffold_pdb = pd.read_csv(DATAPATH / "SCAFFOLD_SMILES_2_PDBhits.csv")
pdb_radial = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
smiles_molfp = pd.read_pickle(DATAPATH / "molfp_df.pkl")

In [4]:
handler = NAAnoPointer(scaffoldpdb=scaffold_pdb, pdb_radial=pdb_radial,
                      smiles_molfp=smiles_molfp,
                      train_pct=0.8, seed=42)

In [5]:
out_folder = "database"
(handler.make_pointerfile(outpath=DATAPATH))

Building Pointer Files: 100%|██████████| 2377/2377 [00:27<00:00, 86.47it/s] 


True

In [6]:
print(handler.train_rows.head(3))
print(handler.train_rows["WINDOW_IDX"].unique())

print(handler.test_rows.head(3))
print(handler.test_rows["WINDOW_IDX"].unique())

print(len(handler.train_rows))
print(len(handler.test_rows))

                                              SMILES PDB_HIT  WINDOW_IDX
0  C(#Cc1ccc(C#CCOc2nsnc2C2CN3CCC2CC3)cc1)COc1nsn...    5YC8           0
1  C(#Cc1ccc(C#CCOc2nsnc2C2CN3CCC2CC3)cc1)COc1nsn...    5YC8           1
2  C(#Cc1ccc(C#CCOc2nsnc2C2CN3CCC2CC3)cc1)COc1nsn...    5YC8           2
[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106]
                                           SMILES PDB_HIT  WINDOW_IDX
0  C(#Cc1ccc2ccccc2n1)c1cc(-n2ncnc2Cc2ccccn2)ncn1    5SKR           0
1  C(#Cc1ccc2ccccc2n1)c1cc(-n2ncnc2Cc2ccccn2)ncn1    5SKR           1
2  C(#Cc1ccc2ccccc2n1)c1cc(-n2ncnc2Cc2ccccn2)ncn1    5SKR      

In [7]:
pdb_radial = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
smiles_molfp = pd.read_pickle(DATAPATH / "molfp_df.pkl")
train_pointer = pd.read_parquet(DATAPATH / "naanobot_training_pointers.parquet")
test_pointer = pd.read_parquet(DATAPATH / "naanobot_test_pointers.parquet")

# once you've run the cells below you can delete the files mentioned above
# I kept them mostly just to inspect and have as an intermediate for transforming into a dictionary

In [8]:
import pyarrow.parquet as pq
import pyarrow as pa

In [9]:
train_pointer = DATAPATH / "naanobot_training_pointers.parquet"

table = pq.read_table(train_pointer)

table = table.set_column(
    table.schema.get_field_index("SMILES"),
    "SMILES",
    table.column("SMILES").dictionary_encode()
)
table = table.set_column(
    table.schema.get_field_index("PDB_HIT"),
    "PDB_HIT",
    table.column("PDB_HIT").dictionary_encode()
)

pq.write_table(table, DATAPATH / "naanobot_training_pointers_dict.parquet", compression="zstd")

In [10]:
test_pointer = DATAPATH / "naanobot_test_pointers.parquet"

table = pq.read_table(test_pointer)

table = table.set_column(
    table.schema.get_field_index("SMILES"),
    "SMILES",
    table.column("SMILES").dictionary_encode()
)
table = table.set_column(
    table.schema.get_field_index("PDB_HIT"),
    "PDB_HIT",
    table.column("PDB_HIT").dictionary_encode()
)

pq.write_table(table, DATAPATH / "naanobot_test_pointers_dict.parquet", compression="zstd")